<a href="https://colab.research.google.com/github/jorgemunozl/vla-test/blob/main/sixth_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Test 17 - Sample Actions

I need to know better how the action expert works. Play with it.

In [1]:
import os

# Set this BEFORE importing pytorch/tensorflow
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

import torch
# Check if it worked (should return 1 if you selected a single GPU)
print(torch.cuda.device_count()) 

1


In [2]:
DS_ID = "NONHUMAN-RESEARCH/pick-and-place-all-fruits-annotated"
PRETRAINED_PATH = "lerobot/pi05_base"

In [3]:
from xhuman.policies.pi05ki.configuration_pi05ki import PI05KIConfig

policy_config = PI05KIConfig(repo_id="none",device="cuda",pretrained_path=PRETRAINED_PATH)

/home/lperez/miniconda3/envs/xhuman_nh/lib/python3.12/site-packages/jaxlib/plugin_support.py:71: RuntimeWarning: JAX plugin jax_cuda12_plugin version 0.5.3 is installed, but it is not compatible with the installed jaxlib version 0.7.2, so it will not be used.
  warnings.warn(


In [4]:
from xhuman.configs.default import LerobotDatasetConfig
from xhuman.configs.default import LerobotDatasetConfig

dataset_config = LerobotDatasetConfig(
    repo_id=DS_ID,
)
dataset_config = LerobotDatasetConfig(
    repo_id=DS_ID,
)

In [5]:
from xhuman.configs.train import TrainPipelineConfigXHUMAN

cfg = TrainPipelineConfigXHUMAN(
    dataset=dataset_config,
    policy=policy_config,
)
cfg.validate()

In [6]:
from xhuman.datasets.factory import make_dataset_xhuman

cfg.dataset.train_with_subtasks = True
dataset = make_dataset_xhuman(cfg)


In [8]:
from xhuman.policies.factory import make_xhuman_policy

policy = make_xhuman_policy(
    cfg=cfg.policy,
    ds_meta=dataset.meta,
)

Loading model from: lerobot/pi05_base
✓ Loaded state dict from model.safetensors


[18:02:24] WARNING  Vision embedding key might need handling:                               ]8;id=753876;file:///home/lperez/main/nh/XHUMAN/xhuman/policies/pi05ki/modeling_pi05ki.py\modeling_pi05ki.py]8;;\:]8;id=133110;file:///home/lperez/main/nh/XHUMAN/xhuman/policies/pi05ki/modeling_pi05ki.py#2027\2027]8;;\
                    paligemma_with_expert.paligemma.model.vision_tower.vision_model.embeddi                        
                    ngs.patch_embedding.bias                                                                       

           WARNING  Vision embedding key might need handling:                               ]8;id=82129;file:///home/lperez/main/nh/XHUMAN/xhuman/policies/pi05ki/modeling_pi05ki.py\modeling_pi05ki.py]8;;\:]8;id=572582;file:///home/lperez/main/nh/XHUMAN/xhuman/policies/pi05ki/modeling_pi05ki.py#2027\2027]8;;\
                    paligemma_with_expert.paligemma.model.vision_tower.vision_model.embeddi                        
                    ngs.patch_embedding.weight                                                                     

Remapped: action_in_proj.bias -> model.action_in_proj.bias
Remapped: action_in_proj.weight -> model.action_in_proj.weight
Remapped: action_out_proj.bias -> model.action_out_proj.bias
Remapped: action_out_proj.weight -> model.action_out_proj.weight
Remapped: paligemma_with_expert.gemma_expert.lm_head.weight -> model.paligemma_with_expert.gemma_expert.lm_head.weight
Remapped: paligemma_with_expert.gemma_expert.model.layers.0.input_layernorm.dense.bias -> model.paligemma_with_expert.gemma_expert.model.layers.0.input_layernorm.dense.bias
Remapped: paligemma_with_expert.gemma_expert.model.layers.0.input_layernorm.dense.weight -> model.paligemma_with_expert.gemma_expert.model.layers.0.input_layernorm.dense.weight
Remapped: paligemma_with_expert.gemma_expert.model.layers.0.mlp.down_proj.weight -> model.paligemma_with_expert.gemma_expert.model.layers.0.mlp.down_proj.weight
Remapped: paligemma_with_expert.gemma_expert.model.layers.0.mlp.gate_proj.weight -> model.paligemma_with_expert.gemma_expe

In [9]:
from xhuman.policies.factory import make_xhuman_pre_post_processors

preprocessor, _ = make_xhuman_pre_post_processors(
        policy_cfg=policy_config,
    )

In [10]:
device = torch.device("cuda")

In [11]:
frame = dataset[0]
frame.keys()

dict_keys(['observation.images.left', 'observation.images.top', 'observation.images.right', 'action', 'observation.state', 'timestamp', 'frame_index', 'episode_index', 'index', 'task_index', 'general_task_index', 'action_is_pad', 'task', 'general_task', 'train_with_subtask'])

In [12]:
frame["observation.images.top"].shape

torch.Size([3, 376, 672])

In [13]:
batch_subtask = preprocessor.subtask(frame)

In [14]:
batch_subtask.keys()

dict_keys(['action', 'next.reward', 'next.done', 'next.truncated', 'info', 'action_is_pad', 'task', 'index', 'task_index', 'episode_index', 'train_with_subtask', 'subtask', 'subtask_tokens', 'observation.images.left', 'observation.images.top', 'observation.images.right', 'observation.state', 'observation.language.tokens', 'observation.language.attention_mask'])

In [15]:
batch_subtask["task"]

['Task: put all the fruits in the basket. Subtask: ']

In [16]:
img, img_mask = policy._preprocess_images(batch_subtask)

In [17]:
tokens , mask = batch_subtask["observation.language.tokens"], batch_subtask["observation.language.attention_mask"]
# img, batch_subtask["observation.images.top"]

In [ ]:
# a = policy.model.sample_subtask(img, img_mask, tokens, mask, 200, 1, 0.1)
# Over Simplifie
actions_chunk = policy.predict_action_chunk(batch_subtask)

In [32]:
actions_chunk.shape

torch.Size([1, 50, 14])

In [35]:
OBS_LANGUAGE_TOKENS = "observation.language.tokens"
OBS_LANGUAGE_ATTENTION_MASK = "observation.language.attention_mask"

images, img_masks = policy._preprocess_images(batch_subtask)
tokens = batch_subtask[f"{OBS_LANGUAGE_TOKENS}"]
masks = batch_subtask[f"{OBS_LANGUAGE_ATTENTION_MASK}"]

from xhuman.policies.utils import make_att_2d_masks

In [37]:
noise = None
num_steps = None

In [ ]:
    if num_steps is None:
        num_steps = policy.model.config.num_inference_steps

        bsize = tokens.shape[0]
        device = tokens.device

        if noise is None:
            # Sample noise with padded dimension as expected by action_in_proj
            actions_shape = (
                bsize,
                policy.model.config.chunk_size,
                policy.model.config.max_action_dim,
            )  # Use config max_action_dim for internal processing
            noise = policy.model.sample_noise(actions_shape, device)

        prefix_embs, prefix_pad_masks, prefix_att_masks = policy.model.embed_prefix(images, img_masks, tokens, masks)
        prefix_att_2d_masks = make_att_2d_masks(prefix_pad_masks, prefix_att_masks)
        prefix_position_ids = torch.cumsum(prefix_pad_masks, dim=1) - 1

        prefix_att_2d_masks_4d = policy.model._prepare_attention_masks_4d(
            prefix_att_2d_masks, dtype=prefix_embs.dtype
        )
        policy.model.paligemma_with_expert.paligemma.language_model.config._attn_implementation = "eager"  # noqa: SLF001

        _, past_key_values = policy.model.paligemma_with_expert.forward(
            attention_mask=prefix_att_2d_masks_4d,
            position_ids=prefix_position_ids,
            past_key_values=None,
            inputs_embeds=[prefix_embs, None],
            use_cache=True,
        )

In [47]:
len(past_key_values[0])

2

In [49]:
past_key_values[0][0].shape

torch.Size([1, 1, 968, 256])

In [53]:
    def denoise_step(
        policy,
        prefix_pad_masks,
        past_key_values,
        x_t,
        timestep,
    ):
        """
        Apply one denoising step of the noise `x_t` at a given timestep.

        Args:
            prefix_pad_masks: (B, seq_len_images + seq_len_lang)
            past_key_values: KV cache for efficient autoregressive generation
            x_t: (B, action_horizon, action_dim) -noise tensor for flowmatching
            timestep: (B,) - current timestep
        """
        suffix_embs, suffix_pad_masks, suffix_att_masks, adarms_cond = (
            policy.model.embed_suffix(x_t, timestep)
        )
        print(suffix_embs.shape)
        suffix_len = suffix_pad_masks.shape[1]
        batch_size = prefix_pad_masks.shape[0]
        prefix_len = prefix_pad_masks.shape[1]

        prefix_pad_2d_masks = prefix_pad_masks[:, None, :].expand(batch_size,
                                                                  suffix_len,
                                                                  prefix_len)
        suffix_att_2d_masks = make_att_2d_masks(suffix_pad_masks,
                                                suffix_att_masks)
        full_att_2d_masks = torch.cat([prefix_pad_2d_masks,
                                       suffix_att_2d_masks], dim=2)

        prefix_offsets = torch.sum(prefix_pad_masks, dim=-1)[:, None]
        position_ids = prefix_offsets + torch.cumsum(suffix_pad_masks, dim=1)
        position_ids -= 1

        full_att_2d_masks_4d = policy.model._prepare_attention_masks_4d(
            full_att_2d_masks, dtype=suffix_embs.dtype
        )
        policy.model.paligemma_with_expert.gemma_expert.model.config._attn_implementation = "eager"  # noqa: SLF001, E501

        # Important, here is the vector field core
        outputs_embeds, _ = policy.model.paligemma_with_expert.forward(
            attention_mask=full_att_2d_masks_4d,
            position_ids=position_ids,
            past_key_values=past_key_values,
            inputs_embeds=[None, suffix_embs],
            use_cache=False,
            adarms_cond=[None, adarms_cond],
        )

        suffix_out = outputs_embeds[1]
        suffix_out = suffix_out[:, -policy.model.config.chunk_size:]
        suffix_out = suffix_out.to(dtype=torch.float32)
        return policy.model.action_out_proj(suffix_out)


In [ ]:
        dt = -1.0 / num_steps
        dt = torch.tensor(dt, dtype=torch.float32, device=device)

        x_t = noise
        print(x_t.shape)
        time = torch.tensor(1.0, dtype=torch.float32, device=device)
        while time >= -dt / 2:
            expanded_time = time.expand(bsize)

            # Define a closure function to properly capture expanded_time
            # This avoids the lambda expression (E731) and loop variable binding (B023) issues
            def denoise_step_partial_call(input_x_t, current_timestep=expanded_time):
                return denoise_step(
                    policy,
                    prefix_pad_masks=prefix_pad_masks,
                    past_key_values=past_key_values,
                    x_t=input_x_t,
                    timestep=current_timestep,
                )

            if policy.model._rtc_enabled():
                print("RTC enabled")
                inference_delay = kwargs.get("inference_delay")
                prev_chunk_left_over = kwargs.get("prev_chunk_left_over")
                execution_horizon = kwargs.get("execution_horizon")

                v_t = policy.model.rtc_processor.denoise_step(
                    x_t=x_t,
                    prev_chunk_left_over=prev_chunk_left_over,
                    inference_delay=inference_delay,
                    time=time,
                    original_denoise_step_partial=denoise_step_partial_call,
                    execution_horizon=execution_horizon,
                )
            else:
                print("RTC disabled")
                v_t = denoise_step_partial_call(x_t)

            # Euler step
            x_t += dt * v_t

            # Record x_t and v_t after Euler step
            if policy.model.rtc_processor is not None and policy.model.rtc_processor.is_debug_enabled():
                policy.model.rtc_processor.track(time=time, x_t=x_t, v_t=v_t)

            time += dt

RTC disabled
torch.Size([1, 50, 1024])
RTC disabled
torch.Size([1, 50, 1024])
RTC disabled
torch.Size([1, 50, 1024])
RTC disabled
torch.Size([1, 50, 1024])
RTC disabled
torch.Size([1, 50, 1024])
RTC disabled
torch.Size([1, 50, 1024])
RTC disabled
torch.Size([1, 50, 1024])
RTC disabled
torch.Size([1, 50, 1024])
RTC disabled
torch.Size([1, 50, 1024])
RTC disabled
torch.Size([1, 50, 1024])


In [ ]:
# It doens't have much sense output a vector with that shape.
x_t.shape

torch.Size([1, 50, 32])